# Try a molecule against the historical record

Nothing to install locally — this runs in the browser.

You give a SMILES string and a series id. The notebook tells you whether the
structure is valid, whether it stayed on the scaffold, whether medicinal chemists
actually made it (and what they measured), and whether the type of change you made
matches the changes that raised potency in that real series.

A molecule cannot be judged in the abstract — *"is this good?"* has no referent.
What this answers is *"is this a good optimization of **that** lead?"*, which the
published series does settle.

**Run the cells in order.** The first one takes a minute.

In [ ]:
!pip install -q rdkit
!pip install -q git+https://github.com/SinanGokmen/leadopt-bench.git
print("ready")

## 1. Which series can you test against?

Each row is a real optimization campaign from a published paper: a starting
compound with a measured potency, and the improved analogs the chemists reached.

In [ ]:
import pandas as pd
from leadopt_bench import metrics as M

pd.set_option("display.max_colwidth", 70)
df = pd.DataFrame(M.available_series())
df[["series_id", "target", "endpoint", "lead_pchembl", "n_targets", "max_gain"]]

## 2. Look at one series' starting compound

Pick a `series_id` from the table above. This draws the lead you will be improving.

In [ ]:
SERIES_ID = "CHEMBL6106211_IC50"  # <-- change this to any series_id above

from rdkit import Chem
from rdkit.Chem import Draw

info = next(s for s in M.available_series() if s["series_id"] == SERIES_ID)
print(f"target:   {info['target']}")
print(f"endpoint: {info['endpoint']}")
print(f"lead:     {info['lead_smiles']}")
print(f"measured: p{info['endpoint']} = {info['lead_pchembl']}")
print(f"chemists reached {info['n_targets']} better compounds, "
      f"best gain {info['max_gain']} log units")
Draw.MolToImage(Chem.MolFromSmiles(info["lead_smiles"]), size=(520, 320))

## 3. Propose a modification

Edit `MY_SMILES` below. Start from the lead printed above and change something.

Filling in `MY_REASONING` is optional, but if you do, the tool also checks whether
the structures your reasoning mentions are actually present in the molecule — that
check is what catches a plausible-sounding rationale about a ring that isn't there.

In [ ]:
MY_SMILES = "CC(C)(C)c1cc(NC(=O)Nc2ccc(Oc3ccnc(-c4cnn(CCCCCCC(=O)NO)c4)c3)cc2)no1"
MY_CHANGE = "shorten the acyl chain"
MY_REASONING = "the hydroxamic acid chelates zinc; the chain should match the channel depth"

score = M.evaluate_smiles(MY_SMILES, SERIES_ID,
                          modification=MY_CHANGE, rationale=MY_REASONING)
print(M.explain(score))

### How to read the verdict

| Line | Meaning |
|---|---|
| `CHEMISTS MADE THIS and it was more potent` | your molecule is in the real series and it worked — the measured value is shown |
| `chemists made this ... but it was not more potent` | it was tried and it did not help |
| `absent from the series` | nobody ever made it. **Not scored** — there is no ground truth, so this is a question for a medicinal chemist |
| `left the scaffold` | further from the lead than the real campaign ever went; this is scaffold hopping, not lead optimization |
| `change-type accuracy` | how much your change overlaps, at functional-group level, with the changes that raised potency in this series. **Read it against the null floor of ~0.53-0.61, not as an absolute** |
| `RATIONALE INCONSISTENT` | your text names a structure that is in neither the lead nor your proposal |

Nothing here is a model's opinion. All of it is RDKit substructure matching and
measured ChEMBL activity.

## 4. See the molecules side by side

In [ ]:
mols, legends = [], []
lead = Chem.MolFromSmiles(info["lead_smiles"])
mols.append(lead); legends.append(f"lead  p={info['lead_pchembl']}")
mine = Chem.MolFromSmiles(MY_SMILES)
if mine is not None:
    mols.append(mine)
    legends.append(f"yours  ({score.category})")
Draw.MolsToGridImage(mols, legends=legends, molsPerRow=2, subImgSize=(420, 300))

## 5. What an LLM proposed, and what survived the filters

Gemma 3 12B was asked the same question on all 32 series: 160 suggestions.
The table below is the queue that reaches a human — valid molecules that kept the
scaffold and whose reasoning is at least about the right compound, but which the
chemists never made. These are exactly the ones only a domain expert can settle.

In [ ]:
import importlib.resources as res

path = res.files("leadopt_bench") / "data/reports/suggestions_final_gemma3_12b.csv"
sug = pd.read_csv(path)

print("all 160 suggestions by category:")
print(sug["category"].value_counts().to_string())

queue = sug[(sug["category"] == "novel_plausible") & (~sug["describes_other_molecule"])]
print(f"\nreaching a human: {len(queue)} of {len(sug)}")
queue[["target", "modification", "rationale", "smiles"]].head(10)

## 6. What the filters threw out, and why

Two thirds of the suggestions never reach a human. Worth seeing what they look like.

In [ ]:
print("--- discarded: not a valid molecule ---")
bad = sug[sug["category"] == "invalid"]
print(bad[["modification", "invalid_reason"]].head(5).to_string(index=False))

print("\n--- discarded: the lead returned unchanged ---")
same = sug[sug["category"] == "unchanged"]
print(f"{len(same)} suggestions described a change and made none")

print("\n--- discarded: reasoning about structures that are not there ---")
ung = sug[sug["describes_other_molecule"] == True]
print(ung["ungrounded_claims"].value_counts().head(8).to_string())
print(f"\n{len(ung)} of {sug['smiles'].notna().sum()} suggestions, e.g.:")
for _, r in ung.head(2).iterrows():
    print(f"\n  claimed: {r['modification']}")
    print(f"  not in the molecule: {r['ungrounded_claims']}")